<a href="https://colab.research.google.com/github/annapatricia/policy-intelligence-platform/blob/main/notebooks/mvp_extracao_poder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 14.6 MB/s eta 0:00:00


#GITHUB

In [2]:
import requests

url = "https://raw.githubusercontent.com/annapatricia/policy-intelligence-platform/main/data/raw/world_bank/education/secondary_education/dropout/WB_PODER_Mexico_2019.pdf"

response = requests.get(url)

with open("WB_PODER_Mexico_2019.pdf", "wb") as f:
    f.write(response.content)

print("PDF baixado com sucesso")

PDF baixado com sucesso


#LENDO O TEXTO

In [3]:
import fitz  # PyMuPDF

pdf_path = "WB_PODER_Mexico_2019.pdf"

document = fitz.open(pdf_path)

full_text = ""

for page_number, page in enumerate(document, start=1):
    text = page.get_text()
    full_text += f"\n--- PAGE {page_number} ---\n"
    full_text += text

print("Número de páginas:", len(document))
print()
print("Primeiros 3000 caracteres:")
print()
print(full_text[:3000])

Número de páginas: 32

Primeiros 3000 caracteres:


--- PAGE 1 ---
Policy Research Working Paper
9085
Addressing High School Dropouts 
with a Scalable Intervention
The Case of PODER
Ciro Avitabile
Janina Cuevas
Rafael de Hoyos
Julian C. Jamison
Education Global Practice 
December 2019
Public Disclosure Authorized
Public Disclosure Authorized
Public Disclosure Authorized
Public Disclosure Authorized

--- PAGE 2 ---
Produced by the Research Support Team
Abstract
The Policy Research Working Paper Series disseminates the findings of work in progress to encourage the exchange of ideas about development 
issues. An objective of the series is to get the findings out quickly, even if the presentations are less than fully polished. The papers carry the 
names of the authors and should be cited accordingly. The findings, interpretations, and conclusions expressed in this paper are entirely those 
of the authors. They do not necessarily represent the views of the International Bank for Reconstruc

#SALVAR EM UM TXT

In [4]:
with open("WB_PODER_Mexico_2019.txt", "w", encoding="utf-8") as f:
    f.write(full_text)

print("Texto salvo com sucesso em WB_PODER_Mexico_2019.txt")

Texto salvo com sucesso em WB_PODER_Mexico_2019.txt


#Onde esta ?

In [5]:
import os

print(os.getcwd())
print(os.listdir())

/content
['.config', 'WB_PODER_Mexico_2019.pdf', 'WB_PODER_Mexico_2019.txt', 'sample_data']


#PREPARANDO O  TEXTO PARA O LLM

#Dividir o texto em blocos

In [6]:
chunk_size = 4000

chunks = [
    full_text[i:i + chunk_size]
    for i in range(0, len(full_text), chunk_size)
]

print("Quantidade de blocos:", len(chunks))
print()
print("Primeiro bloco:")
print()
print(chunks[0][:1500])

Quantidade de blocos: 21

Primeiro bloco:


--- PAGE 1 ---
Policy Research Working Paper
9085
Addressing High School Dropouts 
with a Scalable Intervention
The Case of PODER
Ciro Avitabile
Janina Cuevas
Rafael de Hoyos
Julian C. Jamison
Education Global Practice 
December 2019
Public Disclosure Authorized
Public Disclosure Authorized
Public Disclosure Authorized
Public Disclosure Authorized

--- PAGE 2 ---
Produced by the Research Support Team
Abstract
The Policy Research Working Paper Series disseminates the findings of work in progress to encourage the exchange of ideas about development 
issues. An objective of the series is to get the findings out quickly, even if the presentations are less than fully polished. The papers carry the 
names of the authors and should be cited accordingly. The findings, interpretations, and conclusions expressed in this paper are entirely those 
of the authors. They do not necessarily represent the views of the International Bank for Reconstruction and

#PROMPT usado   pelo  LLM

In [7]:
prompt_template = """
Você é um assistente especializado em análise de políticas públicas.

Analise exclusivamente o texto fornecido.

Extraia as informações abaixo somente quando estiverem explicitamente presentes no texto.
Se uma informação não estiver disponível, retorne null.
Não invente, não complete por conhecimento externo e não faça suposições.

Retorne o resultado em JSON.

Campos:

{
  "country": null,
  "region": null,
  "program_name": null,
  "public_problem": null,
  "target_population": null,
  "intervention": [],
  "implementation_setting": null,
  "delivery_agents": null,
  "evaluation_method": null,
  "sample_size": null,
  "outcomes": [],
  "main_results": [],
  "implementation_problems": [],
  "limitations": [],
  "evidence": []
}

Para cada informação importante, quando possível, inclua em "evidence":
- campo
- trecho do documento
- página

TEXTO DO DOCUMENTO:
"""
print(prompt_template)


Você é um assistente especializado em análise de políticas públicas.

Analise exclusivamente o texto fornecido.

Extraia as informações abaixo somente quando estiverem explicitamente presentes no texto.
Se uma informação não estiver disponível, retorne null.
Não invente, não complete por conhecimento externo e não faça suposições.

Retorne o resultado em JSON.

Campos:

{
  "country": null,
  "region": null,
  "program_name": null,
  "public_problem": null,
  "target_population": null,
  "intervention": [],
  "implementation_setting": null,
  "delivery_agents": null,
  "evaluation_method": null,
  "sample_size": null,
  "outcomes": [],
  "main_results": [],
  "implementation_problems": [],
  "limitations": [],
  "evidence": []
}

Para cada informação importante, quando possível, inclua em "evidence":
- campo
- trecho do documento
- página

TEXTO DO DOCUMENTO:



#TESTAR O PROMPT COM UM BLOCO DE DOCUMENTOS

In [8]:
texto_para_llm = prompt_template + "\n" + chunks[0]

print(texto_para_llm[:6000])


Você é um assistente especializado em análise de políticas públicas.

Analise exclusivamente o texto fornecido.

Extraia as informações abaixo somente quando estiverem explicitamente presentes no texto.
Se uma informação não estiver disponível, retorne null.
Não invente, não complete por conhecimento externo e não faça suposições.

Retorne o resultado em JSON.

Campos:

{
  "country": null,
  "region": null,
  "program_name": null,
  "public_problem": null,
  "target_population": null,
  "intervention": [],
  "implementation_setting": null,
  "delivery_agents": null,
  "evaluation_method": null,
  "sample_size": null,
  "outcomes": [],
  "main_results": [],
  "implementation_problems": [],
  "limitations": [],
  "evidence": []
}

Para cada informação importante, quando possível, inclua em "evidence":
- campo
- trecho do documento
- página

TEXTO DO DOCUMENTO:


--- PAGE 1 ---
Policy Research Working Paper
9085
Addressing High School Dropouts 
with a Scalable Intervention
The Case of P

## Próximo passo: integração com LLM

A próxima etapa deste MVP será integrar um modelo de linguagem (LLM) ao pipeline.

O objetivo será enviar os blocos de texto extraídos automaticamente do documento para o LLM, utilizando um prompt estruturado.

O LLM deverá identificar e extrair informações relevantes para a análise de políticas públicas, como:

- país e região;
- problema público;
- público-alvo;
- intervenção;
- método de avaliação;
- tamanho da amostra;
- resultados;
- problemas de implementação;
- limitações;
- evidências encontradas no documento.

A saída esperada será em formato JSON, seguindo a estrutura definida para o `PolicyCase`.

Fluxo previsto:

PDF
→ Python
→ Extração de texto
→ Divisão em blocos (chunks)
→ Prompt estruturado
→ LLM
→ JSON
→ Validação humana
→ Base curada

Nesta versão inicial do MVP, a chamada automática ao LLM não foi implementada. A integração ficará como próxima etapa de desenvolvimento.

A validação humana continuará sendo necessária antes que um caso seja considerado validado e incorporado à base oficial.

## Próximo passo: integração com LLM

A próxima etapa deste MVP será integrar um modelo de linguagem (LLM) ao pipeline.

O objetivo será enviar os blocos de texto extraídos automaticamente do documento para o LLM, utilizando um prompt estruturado.

O LLM deverá identificar e extrair informações relevantes para a análise de políticas públicas, como:

- país e região;
- problema público;
- público-alvo;
- intervenção;
- método de avaliação;
- tamanho da amostra;
- resultados;
- problemas de implementação;
- limitações;
- evidências encontradas no documento.

A saída esperada será em formato JSON, seguindo a estrutura definida para o `PolicyCase`.

Fluxo previsto:

PDF
→ Python
→ Extração de texto
→ Divisão em blocos (chunks)
→ Prompt estruturado
→ LLM
→ JSON
→ Validação humana
→ Base curada

Nesta versão inicial do MVP, a chamada automática ao LLM não foi implementada. A integração ficará como próxima etapa de desenvolvimento.

A validação humana continuará sendo necessária antes que um caso seja considerado validado e incorporado à base oficial.